In [1]:
import csv
import logging
import os
import time
import re
from typing import Union
import numpy as np
import pandas as pd
from pathlib import Path
import json

from aind_codeocean_pipeline_monitor.models import (CaptureSettings,
                                                    PipelineMonitorSettings)
from aind_data_access_api.document_db import MetadataDbClient
from codeocean import CodeOcean
from codeocean.computation import (ComputationState, DataAssetsRunParam,
                                   RunParams)
from dataclasses_json import dataclass_json

from lamf_analysis.code_ocean import docdb_utils
from lamf_analysis.code_ocean import capsule_data_utils as cdu
from lamf_analysis.code_ocean import code_ocean_utils as cou

%load_ext autoreload
%autoreload 2



In [2]:
subject_ids = [755252, 767018, 767022, 783551, 785054, 782149, 788406, 790322, 800792, 800995, 804363, 804670] # all Slc32a1;Oi1 collected so far


In [3]:
def get_attach_df(subject_id):
    # including single-cell-zdrift
    session_infos = docdb_utils.get_session_infos_from_docdb(subject_id, filter_test_data=True)
    processed_infos = docdb_utils.get_processed_data_info(subject_id).sort_values('long_window').drop_duplicates(subset=['raw_name'])
    merged_df = processed_infos.merge(session_infos, left_on='raw_name', right_on='raw_asset_name', how='left')

    sczdrift_df = docdb_utils.get_derived_data_assets(subject_id, 'single-cell-zdrift-qc')
    sczdrift_df.rename(columns={'derived_name': 'single_cell_zdrift_derived_name',
                                'derived_asset_id': 'single_cell_zdrift_derived_asset_id'}, inplace=True)
    merged_df = merged_df.merge(sczdrift_df[['raw_name',
                                            'single_cell_zdrift_derived_name',
                                            'single_cell_zdrift_derived_asset_id']],
                                on='raw_name', how='inner')
    # attach_df = merged_df[merged_df.session_type.str.contains('OPHYS_')]
    return merged_df

In [5]:
subject_id = subject_ids[-1]
merged_df = get_attach_df(subject_id)
attach_asset_ids = merged_df['raw_asset_id'].tolist() + merged_df['processed_asset_id'].tolist() + merged_df['single_cell_zdrift_derived_asset_id'].tolist()
cou.attach_assets(attach_asset_ids)
# processed_infos

asset_id: 2d08fe8b-02b1-4c0d-9ea7-7e284ee42480 - mount_state: 
asset_id: 94b6f2fc-60fc-41ef-bcf1-22ee34734641 - mount_state: 
asset_id: da3c18b4-ff89-4c4b-8992-590d724c38c1 - mount_state: 
asset_id: d29888d1-5519-4967-947e-490b879c7c5e - mount_state: 
asset_id: 6be07a5d-04b2-4de5-a8ae-13e3981c1f3b - mount_state: 
asset_id: a0ec7e41-1e04-423e-9b8e-c1ceee844fce - mount_state: 
asset_id: 20faafd4-b93e-46d9-a078-cd2d000f8404 - mount_state: 
asset_id: a85c3be1-73c8-4e17-b2a4-31c1fd130c9c - mount_state: 
asset_id: 62927066-2c95-4287-88dd-7a9e16799735 - mount_state: 
asset_id: 1857358d-08a0-4cb8-8398-f6c745c74172 - mount_state: 
asset_id: ffedea60-2555-47ae-870b-91fd776de27f - mount_state: 
asset_id: 96296bb4-098a-4cdd-b551-5065ff57716a - mount_state: 
asset_id: 52bcbd02-f9cc-45de-bbd2-9c62c6484f07 - mount_state: 
asset_id: 52bcbd02-f9cc-45de-bbd2-9c62c6484f07 - mount_state: 
asset_id: 0255a8a2-f701-415d-b2ed-fc84d22b363f - mount_state: 
asset_id: 0255a8a2-f701-415d-b2ed-fc84d22b363f - mount_